In [6]:
import numpy as np
import pandas as pd
from utils import load_embeddings, load_embeddings_vec, cosine_sim, plot_pca

In [7]:
print("Loading aligned embeddings (MUSE)")
aligned_types = ["w2v", "ft", "glv", "ohe", "tfidf"]
embeddings_aligned = {et: load_embeddings(et) for et in aligned_types}

print("Loading unaligned embeddings (.vec)...")
unaligned_types = ["w2v", "ft", "glv", "ohe", "tfidf"]
embeddings_unaligned = {et: load_embeddings_vec(et) for et in unaligned_types}

print(f"Aligned: {list(embeddings_aligned.keys())}")
print(f"Unaligned: {list(embeddings_unaligned.keys())}")


Loading aligned embeddings (MUSE)
Loading unaligned embeddings (.vec)...


In [ ]:
# Synonyms
en_synonyms = [
    ('happy', 'joyful'), ('big', 'large'), ('smart', 'intelligent'), 
    ('fast', 'quick'), ('beautiful', 'pretty'), ('angry', 'furious')
]
fr_synonyms = [
    ('heureux', 'joyeux'), ('grand', 'gros'), ('intelligent', 'brillant'), 
    ('rapide', 'vite'), ('beau', 'joli'), ('furieux', 'enragé')
]

# Antonyms
en_antonyms = [
    ('happy', 'sad'), ('big', 'small'), ('hot', 'cold'), 
    ('good', 'bad'), ('fast', 'slow'), ('light', 'dark')
]
fr_antonyms = [
    ('heureux', 'triste'), ('grand', 'petit'), ('chaud', 'froid'), 
    ('bon', 'mauvais'), ('rapide', 'lent'), ('clair', 'sombre')
]

# Polysemy (English only)
en_polysemy = [
    ('bank', 'money'), ('bank', 'river'),
    ('light', 'dark'), ('light', 'heavy'),
    ('bat', 'baseball'), ('bat', 'animal')
]

# OOV - test n-gram capability with compound/derived words
oov_tests = {
    'base': ['cat', 'walk', 'run', 'play'],
    'derived': ['walking', 'running', 'playing', 'walked'],
    'compound': ['catwalking', 'jaywalking', 'moonwalking', 'sleepwalking'],
    'rare_derived': ['unbelievable', 'misconception', 'preprocessing', 'counterintuitive']
}

Testing W2V aligned translations:
  cat vs chat: 0.997
  dog vs chien: 0.997
  house vs maison: 0.994
  car vs voiture: 0.989
  book vs livre: 0.979


In [ ]:
def compute_avg_sim(word_pairs, emb, lang1='en', lang2='en'):
    """Compute average similarity, filtering None values"""
    sims = [cosine_sim(w1, w2, emb, lang1, lang2) for w1, w2 in word_pairs]
    valid = [s for s in sims if s is not None]
    return np.mean(valid) if valid else None

W2V: avg=0.9913, min=0.9792, max=0.9974
FT: avg=0.9816, min=0.9560, max=0.9942
GLV: avg=0.5371, min=0.3933, max=0.6176
OHE: avg=0.6651, min=0.0170, max=1.0000
TFIDF: avg=0.7196, min=0.0192, max=1.0000


In [ ]:
results = []

# Aligned
for emb_type in aligned_types:
    emb = embeddings_aligned[emb_type]
    results.append({
        'Type': 'Aligned',
        'Model': emb_type.upper(),
        'EN_Syn': f"{compute_avg_sim(en_synonyms, emb, 'en', 'en'):.3f}",
        'EN_Ant': f"{compute_avg_sim(en_antonyms, emb, 'en', 'en'):.3f}",
        'FR_Syn': f"{compute_avg_sim(fr_synonyms, emb, 'fr', 'fr'):.3f}",
        'FR_Ant': f"{compute_avg_sim(fr_antonyms, emb, 'fr', 'fr'):.3f}"
    })

# Unaligned
for emb_type in unaligned_types:
    emb = embeddings_unaligned[emb_type]
    en_syn = compute_avg_sim(en_synonyms, emb, 'en', 'en')
    en_ant = compute_avg_sim(en_antonyms, emb, 'en', 'en')
    fr_syn = compute_avg_sim(fr_synonyms, emb, 'fr', 'fr')
    fr_ant = compute_avg_sim(fr_antonyms, emb, 'fr', 'fr')
    
    results.append({
        'Type': 'Unaligned',
        'Model': emb_type.upper(),
        'EN_Syn': f"{en_syn:.3f}" if en_syn else "N/A",
        'EN_Ant': f"{en_ant:.3f}" if en_ant else "N/A",
        'FR_Syn': f"{fr_syn:.3f}" if fr_syn else "N/A",
        'FR_Ant': f"{fr_ant:.3f}" if fr_ant else "N/A"
    })

syn_ant_df = pd.DataFrame(results)
print("SYNONYMS vs ANTONYMS")
syn_ant_df

In [ ]:
polysemy_results = []

for emb_type in unaligned_types:
    emb = embeddings_unaligned[emb_type]
    avg_sim = compute_avg_sim(en_polysemy, emb, 'en', 'en')
    polysemy_results.append({
        'Type': 'Unaligned',
        'Model': emb_type.upper(),
        'Avg_Similarity': f"{avg_sim:.3f}" if avg_sim else "N/A"
    })

polysemy_df = pd.DataFrame(polysemy_results)
print("\nPOLYSEMY (ambiguous word pairs)")
polysemy_df

In [ ]:
def count_found(words, emb, lang='en'):
    """Count how many words are found"""
    return sum(1 for w in words if cosine_sim(w, w, emb, lang, lang) is not None)

oov_results = []

for emb_type in unaligned_types:
    emb = embeddings_unaligned[emb_type]
    row = {'Type': 'Unaligned', 'Model': emb_type.upper()}
    for category, words in oov_tests.items():
        row[category.replace('_', ' ').title()] = f"{count_found(words, emb)}/{len(words)}"
    oov_results.append(row)

oov_df = pd.DataFrame(oov_results)
print("\nOOV HANDLING (n-gram capability)")
oov_df

In [ ]:
translation_pairs = [
    ('cat', 'chat'), ('dog', 'chien'), ('house', 'maison'),
    ('car', 'voiture'), ('book', 'livre'), ('water', 'eau')
]
random_pairs_en = [('cat', 'dog'), ('house', 'car'), ('book', 'water')]

space_results = []

# Aligned - cross-lingual and intra-lingual
for emb_type in aligned_types:
    emb = embeddings_aligned[emb_type]
    cross = compute_avg_sim(translation_pairs, emb, 'en', 'fr')
    intra = compute_avg_sim(random_pairs_en, emb, 'en', 'en')
    
    space_results.append({
        'Type': 'Aligned',
        'Model': emb_type.upper(),
        'Cross-Lingual': f"{cross:.3f}" if cross else "N/A",
        'Intra-Lingual': f"{intra:.3f}" if intra else "N/A"
    })

# Unaligned - only intra-lingual (different spaces)
for emb_type in unaligned_types:
    emb = embeddings_unaligned[emb_type]
    intra = compute_avg_sim(random_pairs_en, emb, 'en', 'en')
    
    space_results.append({
        'Type': 'Unaligned',
        'Model': emb_type.upper(),
        'Cross-Lingual': 'N/A',
        'Intra-Lingual': f"{intra:.3f}" if intra else "N/A"
    })

space_df = pd.DataFrame(space_results)
print("\nCROSS-LINGUAL vs INTRA-LINGUAL SIMILARITY")
space_df


In [ ]:
pairs_pca = [('cat', 'chat'), ('dog', 'chien'), ('house', 'maison'), 
             ('car', 'voiture'), ('book', 'livre')]

for emb_type in aligned_types:
    emb = embeddings_aligned[emb_type]
    plot_pca(emb, pairs_pca, f"{emb_type.upper()} - ALIGNED")